In [1]:
!git clone https://github.com/trextrader/hotdogornot

Cloning into 'hotdogornot'...
remote: Enumerating objects: 16247, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (250/250), done.
remote: Total 16247 (delta 125), reused 225 (delta 51), pack-reused 15943 (from 2)
Receiving objects: 100% (16247/16247), 545.22 MiB | 40.48 MiB/s, done.
Resolving deltas: 100% (2213/2213), done.
Updating files: 100% (14465/14465), done.


In [2]:
cd hotdogornot/

/content/hotdogornot


In [3]:
cd training

/content/hotdogornot/training


In [4]:
# @title
!pip install -e ".[dev]"

Obtaining file:///content/hotdogornot/training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 82.0 MB/s eta 

In [11]:
!git pull

Already up to date.


In [ ]:
%cd /content/hotdogornot
!ls training/data/labeled/embedder
!python -c "import sys; sys.path.insert(0,'training'); from rfconnectorai.data.classes import class_names; print('classes:', class_names('training/configs/classes.yaml'))"

In [7]:
%%shell
  set -euxo pipefail
  cd /content/hotdogornot
  export PYTHONPATH=/content/hotdogornot/training
  export PYTHONUNBUFFERED=1

  python -u -m rfconnectorai.data.audit \
      --data-dir training --out docs/DATASET_AUDIT.md

  echo "----- audit summary -----"
  head -40 docs/DATASET_AUDIT.md

  python -u -m rfconnectorai.data.crop_instances \
      --input training/data/labeled/embedder \
      --manifest datasets/rfconnectors/instances.jsonl \
      --out datasets/rfconnectors/crops \
      --mode whole-image --base-dir training

  wc -l datasets/rfconnectors/instances.jsonl

+ cd /content/hotdogornot
+ export PYTHONPATH=/content/hotdogornot/training
+ PYTHONPATH=/content/hotdogornot/training
+ export PYTHONUNBUFFERED=1
+ PYTHONUNBUFFERED=1
+ python -u -m rfconnectorai.data.audit --data-dir training --out docs/DATASET_AUDIT.md
audit report: docs/DATASET_AUDIT.md
audit json:   docs/DATASET_AUDIT.json
+ echo '----- audit summary -----'
----- audit summary -----
+ head -40 docs/DATASET_AUDIT.md
# Dataset Audit

- Generated: `2026-05-10T14:58:33.412372+00:00`
- Data dir: `/content/hotdogornot/training`
- Roots audited: 5

## Summary By Root

| Root | Images | Videos | Synthetic | Holdout | Reference | Multi-conn hint | Unreadable |
|---|---:|---:|---:|---:|---:|---:|---:|
| `/content/hotdogornot/training/Images` | 55 | 0 | 0 | 0 | 0 | 1 | 0 |
| `/content/hotdogornot/training/data/labeled` | 13850 | 0 | 0 | 0 | 0 | 0 | 0 |
| `/content/hotdogornot/training/data/test_holdout` | 8 | 0 | 0 | 8 | 0 | 0 | 0 |
| `/content/hotdogornot/training/data/reference` | 2 | 0 | 

In [ ]:
%%shell
  set -euxo pipefail
  cd /content/hotdogornot
  export PYTHONPATH=/content/hotdogornot/training
  export PYTHONUNBUFFERED=1

  python -u -m rfconnectorai.data.build_yolo_dataset \
      --input datasets/rfconnectors/instances.jsonl \
      --out datasets/rfconnectors --base-dir training \
      --single-class \
      --taxonomy training/rfconnectorai/specs/connectors.yaml

  cat datasets/rfconnectors/data.yaml   # expect nc: 1, names: [connector]

In [ ]:
%%shell
# Stage 1: single-class connector localizer (data.yaml is now nc=1).
set -euxo pipefail
cd /content/hotdogornot
export PYTHONPATH=/content/hotdogornot/training
export PYTHONUNBUFFERED=1

python -u -m rfconnectorai.detector.train_yolo \
    --data datasets/rfconnectors/data.yaml --model yolo11n.pt \
    --epochs 5 --imgsz 640 --batch 96 --device 0 \
    --dataset-lock datasets/rfconnectors/dataset.lock.json \
    --out reports/experiments/detector_run_full \
    --artifact-out models/detector


In [ ]:
%%shell
set -euxo pipefail
cd /content/hotdogornot
export PYTHONPATH=/content/hotdogornot/training
export PYTHONUNBUFFERED=1

python -u -m rfconnectorai.classifier.train \
    --data-dir training/data/labeled/embedder \
    --out-dir models/connector_classifier_phase1 \
    --architecture efficientnet_v2_s --input-size 384 \
    --epochs 30 --batch-size 32 --lr 3e-4

In [ ]:
%%shell
set -euxo pipefail
cd /content/hotdogornot
export PYTHONPATH=/content/hotdogornot/training
export PYTHONUNBUFFERED=1

# Curated-only dir = the clean 2026-05-14 field set. The embedder folder
# already contains exactly the 9 curated classes in a fresh clone of this
# snapshot; if legacy frames are also present, point --data-dir at a
# curated-only copy. Here we fine-tune from the Phase-1 checkpoint.
python -u -m rfconnectorai.classifier.train \
    --data-dir training/data/labeled/embedder \
    --out-dir models/connector_classifier \
    --architecture efficientnet_v2_s --input-size 384 \
    --init-weights models/connector_classifier_phase1/weights.pt \
    --epochs 12 --batch-size 32 --lr 2e-5

In [ ]:
import sys, json
sys.path.insert(0, "training")
from rfconnectorai.eval.nine_class_report import build_report, render_markdown
# y_true / y_pred are produced by running the Phase-2 model over the
# curated holdout split; see rfconnectorai.classifier.predict for the
# inference helper. Persist the report for the run record:
# report = build_report(y_true, y_pred, class_names)
# open("reports/experiments/classifier_9class/REPORT.md","w").write(render_markdown(report))
print("eval helpers imported; wire y_true/y_pred from the curated holdout.")

In [ ]:
!python -m rfconnectorai.classifier.export_onnx \
    --model-dir models/connector_classifier \
    --output models/connector_classifier/classifier.onnx

!zip -r /content/classifier_9class.zip \
    models/connector_classifier/ \
    reports/experiments/classifier_9class/ 2>/dev/null || true

from google.colab import files
files.download('/content/classifier_9class.zip')

In [16]:
!zip -r /content/detector_run_full.zip \
    /content/hotdogornot/reports/experiments/detector_run_full/ \
    /content/hotdogornot/models/detector/best.pt

from google.colab import files
files.download('/content/detector_run_full.zip')



updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/ (stored 0%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/labels.jpg (deflated 55%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/confusion_matrix.png (deflated 29%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/train_batch1.jpg (deflated 10%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/train_batch2.jpg (deflated 20%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/confusion_matrix_normalized.png (deflated 26%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/BoxP_curve.png (deflated 14%)
updating: content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/val_batch2_pred.jpg (deflated 20%)
updating: content/hotdogornot

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Check which runs exist and their sizes
!ls -la /content/hotdogornot/reports/experiments/
!wc -l /content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv
!cat /content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv


total 16
drwxr-xr-x 4 root root 4096 May 10 16:02 .
drwxr-xr-x 3 root root 4096 May 10 14:59 ..
drwxr-xr-x 3 root root 4096 May 10 15:39 detector_run_001
drwxr-xr-x 3 root root 4096 May 10 16:02 detector_run_full
6 /content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv
epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
1,272.308,0.22292,1.81176,1.00814,0.45142,0.47871,0.53226,0.37769,0.593,2.22004,0.63305,0.000472227,0.000472227,0.000472227
2,496.66,0.1015,0.84327,0.91316,0.80668,0.77106,0.90102,0.86376,0.25851,1.08784,0.33686,0.000760745,0.000760745,0.000760745
3,718.536,0.10006,0.6856,0.90861,0.45249,0.52139,0.48401,0.41399,0.46543,3.18782,0.63689,0.000860636,0.000860636,0.000860636
4,936.578,0.08227,0.57937,0.90251,0.90989,0.83175,0.95982,0.94085,0.26688,0.62356,0.35002,0.000580174,0.000580174,0.000580174
5